In [4]:
import json
import statistics
import os
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

def calculate_statistics(file_path: str):
    
    violations: List[float] = []
    number_of_strong: int = 0

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data: List[Dict[str, Any]] = json.load(f)

    except Exception as e:
        print(f"Une erreur inattendue est survenue lors de la lecture : {e}")
        return
    for item in data:
        violation_metric: float = item.get("violation_metric")
        is_strong: bool = item.get("strong", False)
        violations.append(violation_metric)
        if is_strong:
            number_of_strong += 1
    if not violations:
        print("Aucune donnée de violation valide n'a été trouvée. Impossible de calculer les statistiques.")
        return

    try:
        mean_violation = statistics.mean(violations)
        
        percentage_strong = 100 * number_of_strong / len(violations)
        
        print(f"--- Analyse de {file_path} ---")
        print(f"Nombre de violations fortes (>0.2) : {number_of_strong}")
        print(f"Violation moyenne (statistics.mean) : {mean_violation:.4f}")
        print(f"Pourcentage de violations fortes : {percentage_strong:.2f}%")
        print("-" * 30)

    except Exception as e:
        print(f"Une erreur inattendue est survenue lors des calculs : {e}")
        

def process_file(file_path: str) -> Tuple[Optional[float], Optional[float]]:
    """
    Lit un fichier JSON et retourne (pourcentage_strong, mean_violation).
    Retourne (None, None) si le fichier n'existe pas ou est vide.
    """
    violations: List[float] = []
    number_of_strong: int = 0

    try:
        if not os.path.exists(file_path):
            return None, None
            
        with open(file_path, 'r', encoding='utf-8') as f:
            data: List[Dict[str, Any]] = json.load(f)

        for item in data:
            violation_metric: float = item.get("violation_metric")
            is_strong: bool = item.get("strong", False)
            
            if violation_metric is not None:
                violations.append(violation_metric)
            if is_strong:
                number_of_strong += 1
        
        if not violations:
            return None, None

        mean_violation = statistics.mean(violations)
        percentage_strong = 100 * number_of_strong / len(violations)
        
        return percentage_strong, mean_violation

    except Exception as e:
        print(f"Erreur lors du traitement de {file_path}: {e}")
        return None, None

def generate_study_results_table(base_dir: str = "results") -> pd.DataFrame:
    """
    Génère un DataFrame Pandas résumant les résultats de l'étude,
    mimant la structure du tableau Markdown final.
    """
    
    # 1. Définition des catégories (Colonnes) et leurs chemins/préfixes
    categories = {
        "Negation": ("negated_pairs", "negated"),
        "Paraphrasing": ("paraphrases", "paraphrase"),
        "Monotonicity": ("monotonic_sequence", "monotonic_sequence"),
        "Bayes' rule": ("bayes", "bayes")
    }

    # 2. Définition des modèles (Lignes) et leurs suffixes de fichiers
    models_config = [
        ("GPT-3.5-turbo (temp=0)", "gpt-3.5_T-0.0"),
        ("GPT-3.5-turbo (temp=0.5)", "gpt-3.5_T-0.5"),
        ("GPT-4 (temp=0)", "gpt-4_T-0.0"),
        ("GPT-4 (temp=0.5)", "gpt-4_T-0.5"),
    ]

    rows = []

    # 3. Itération pour construire les lignes
    for model_display_name, file_suffix in models_config:
        row_data = {"Model": model_display_name}
        
        for cat_name, (folder, file_prefix) in categories.items():
            # Construction du chemin complet
            # ex: results/negated_pairs/output_negated_gpt-3.5_T-0.0.json
            file_path = os.path.join(
                base_dir, 
                folder, 
                f"output_{file_prefix}_{file_suffix}.json"
            )
            
            pct_strong, mean_val = process_file(file_path)
            
            # Formatage des colonnes
            col_pct = f"{cat_name} (>0.2)"
            col_mean = f"{cat_name} (Mean)"
            
            if pct_strong is not None:
                row_data[col_pct] = f"{pct_strong:.1f}%"
                row_data[col_mean] = f"{mean_val:.2f}"
            else:
                # Gestion des fichiers manquants (ex: Monotonicity dans ton log)
                row_data[col_pct] = "N/A"
                row_data[col_mean] = "N/A"
        
        rows.append(row_data)

    # 4. Création du DataFrame
    df = pd.DataFrame(rows)
    
    # Réorganiser les colonnes pour correspondre exactement à l'ordre souhaité
    desired_order = ["Model"]
    for cat in categories.keys():
        desired_order.append(f"{cat} (>0.2)")
        desired_order.append(f"{cat} (Mean)")
        
    return df[desired_order]

### Negated Pairs

In [12]:
files = ["negated_gpt-3.5_T-0.0","negated_gpt-3.5_T-0.5","negated_gpt-4_T-0.0","negated_gpt-4_T-0.5"]
for f in files:
        calculate_statistics(f"results/negated_pairs/output_{f}.json")

--- Analyse de results/negated_pairs/output_negated_gpt-3.5_T-0.0.json ---
Nombre de violations fortes (>0.2) : 31
Violation moyenne (statistics.mean) : 0.2664
Pourcentage de violations fortes : 48.44%
------------------------------
--- Analyse de results/negated_pairs/output_negated_gpt-3.5_T-0.5.json ---
Nombre de violations fortes (>0.2) : 36
Violation moyenne (statistics.mean) : 0.2942
Pourcentage de violations fortes : 55.38%
------------------------------
--- Analyse de results/negated_pairs/output_negated_gpt-4_T-0.0.json ---
Nombre de violations fortes (>0.2) : 20
Violation moyenne (statistics.mean) : 0.2127
Pourcentage de violations fortes : 30.30%
------------------------------
--- Analyse de results/negated_pairs/output_negated_gpt-4_T-0.5.json ---
Nombre de violations fortes (>0.2) : 20
Violation moyenne (statistics.mean) : 0.2414
Pourcentage de violations fortes : 30.30%
------------------------------


### Bayes

In [13]:
files = ["bayes_gpt-3.5_T-0.0","bayes_gpt-3.5_T-0.5","bayes_gpt-4_T-0.0","bayes_gpt-4_T-0.5"]
for f in files:
        calculate_statistics(f"results/bayes/output_{f}.json")

--- Analyse de results/bayes/output_bayes_gpt-3.5_T-0.0.json ---
Nombre de violations fortes (>0.2) : 4
Violation moyenne (statistics.mean) : 0.2343
Pourcentage de violations fortes : 80.00%
------------------------------
--- Analyse de results/bayes/output_bayes_gpt-3.5_T-0.5.json ---
Nombre de violations fortes (>0.2) : 5
Violation moyenne (statistics.mean) : 0.4827
Pourcentage de violations fortes : 100.00%
------------------------------
--- Analyse de results/bayes/output_bayes_gpt-4_T-0.0.json ---
Nombre de violations fortes (>0.2) : 4
Violation moyenne (statistics.mean) : 0.4883
Pourcentage de violations fortes : 80.00%
------------------------------
--- Analyse de results/bayes/output_bayes_gpt-4_T-0.5.json ---
Nombre de violations fortes (>0.2) : 4
Violation moyenne (statistics.mean) : 0.3858
Pourcentage de violations fortes : 80.00%
------------------------------


### Paraphrasing

In [15]:
files = ["paraphrase_gpt-3.5_T-0.0","paraphrase_gpt-3.5_T-0.5","paraphrase_gpt-4_T-0.0","paraphrase_gpt-4_T-0.5"]
for f in files:
        calculate_statistics(f"results/paraphrases/output_{f}.json")

--- Analyse de results/paraphrases/output_paraphrase_gpt-3.5_T-0.0.json ---
Nombre de violations fortes (>0.2) : 3
Violation moyenne (statistics.mean) : 0.2980
Pourcentage de violations fortes : 60.00%
------------------------------
--- Analyse de results/paraphrases/output_paraphrase_gpt-3.5_T-0.5.json ---
Nombre de violations fortes (>0.2) : 1
Violation moyenne (statistics.mean) : 0.1600
Pourcentage de violations fortes : 20.00%
------------------------------
--- Analyse de results/paraphrases/output_paraphrase_gpt-4_T-0.0.json ---
Nombre de violations fortes (>0.2) : 3
Violation moyenne (statistics.mean) : 0.2400
Pourcentage de violations fortes : 60.00%
------------------------------
--- Analyse de results/paraphrases/output_paraphrase_gpt-4_T-0.5.json ---
Nombre de violations fortes (>0.2) : 3
Violation moyenne (statistics.mean) : 0.1740
Pourcentage de violations fortes : 60.00%
------------------------------


### Monotonic_sequence

In [7]:
files = ["monotonic_sequence_gpt-3.5_T-0.0","monotonic_sequence_gpt-3.5_T-0.5","monotonic_sequence_gpt-4_T-0.0","monotonic_sequence_gpt-4_T-0.5"]
for f in files:
        calculate_statistics(f"results/monotonic_sequence/output_{f}.json")

--- Analyse de results/monotonic_sequence/output_monotonic_sequence_gpt-3.5_T-0.0.json ---
Nombre de violations fortes (>0.2) : 0
Violation moyenne (statistics.mean) : 0.0731
Pourcentage de violations fortes : 0.00%
------------------------------
--- Analyse de results/monotonic_sequence/output_monotonic_sequence_gpt-3.5_T-0.5.json ---
Nombre de violations fortes (>0.2) : 1
Violation moyenne (statistics.mean) : 0.1880
Pourcentage de violations fortes : 50.00%
------------------------------
--- Analyse de results/monotonic_sequence/output_monotonic_sequence_gpt-4_T-0.0.json ---
Nombre de violations fortes (>0.2) : 1
Violation moyenne (statistics.mean) : 0.2750
Pourcentage de violations fortes : 50.00%
------------------------------
--- Analyse de results/monotonic_sequence/output_monotonic_sequence_gpt-4_T-0.5.json ---
Nombre de violations fortes (>0.2) : 1
Violation moyenne (statistics.mean) : 0.1282
Pourcentage de violations fortes : 25.00%
------------------------------


### Resultats de l'étude (tirés du papier)

| Model | Negation (>0.2) | Negation (Mean) | Paraphrasing (>0.2) | Paraphrasing (Mean) | Monotonicity (>0.2) | Monotonicity (Mean) | Bayes’ rule (>0.2) | Bayes’ rule (Mean) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| GPT-3.5-turbo (temp=0) | 52.6% | 0.34 | 30.8% | 0.21 | 42.0% | 0.23 | 68.6% | 0.28 |
| GPT-3.5-turbo (temp=0.5) | 58.9% | 0.31 | 22.1% | 0.16 | 26.0% | 0.14 | 64.7% | 0.24 |
| GPT-4 (temp=0) | 10.9% | 0.10 | 12.5% | 0.13 | 16.0% | 0.11 | 58.8% | 0.25 |
| GPT-4 (temp=0.5) | 8.6% | 0.09 | 14.4% | 0.13 | 12.0% | 0.09 | 74.5% | 0.27 |

Table: Mean violation magnitude and fraction of “strong” violations (with value above ε = 0.2)

### Résultats de la reproductibilité

In [6]:
df_results = generate_study_results_table()
display(df_results)
# df_results.to_csv("resultats_etude.csv", index=False)

,Model,Negation (>0.2),Negation (Mean),Paraphrasing (>0.2),Paraphrasing (Mean),Monotonicity (>0.2),Monotonicity (Mean),Bayes' rule (>0.2),Bayes' rule (Mean)
0,GPT-3.5-turbo (temp=0),48.4%,0.27,60.0%,0.30,0.0%,0.07,80.0%,0.23
1,GPT-3.5-turbo (temp=0.5),55.4%,0.29,20.0%,0.16,50.0%,0.19,100.0%,0.48
2,GPT-4 (temp=0),30.3%,0.21,60.0%,0.24,50.0%,0.28,80.0%,0.49
3,GPT-4 (temp=0.5),30.3%,0.24,60.0%,0.17,25.0%,0.13,80.0%,0.39


### Compraison Etude originale/Etude reproduite

## TODO